# Calibrating the front camera's yaw / pitch lever arm

Seen from the front, the head's **roll** is an in-plane rotation, so Kinefly2
measures it exactly — no rotation centre, no scale factor. The other two
rotations are seen end-on and show up only as a *shift* of the head:

| channel | what it is | logged as |
|---|---|---|
| roll | in-plane angle, exact | `front/head_angle` (rad) |
| yaw | horizontal shift | `front/head_dx_corr` (px) |
| pitch | vertical shift | `front/head_dy_corr` (px) |

Both shifts scale by the **same** constant — the head's anterior–posterior
depth from the neck joint, in pixels. Put the neck at the origin with the
landmark centroid at (lateral, depth, height): a small yaw moves the centroid
sideways by `-yaw * depth`, a small pitch moves it vertically by
`+pitch * depth`. The lateral and height offsets cancel out of both. **So one
number converts both channels.**

That depth points straight down the front camera's optical axis, so a frontal
view can never measure it. This notebook fits it against a reference that can:
the bottom camera's head angle (yaw is in-plane *there*), an external rotation
stage, or spontaneous yaw pooled over long recordings.

Nothing here runs in real time. The pixel channels are what get recorded, so
the constant stays revisable and can be re-applied to old data — which is why
it is deliberately not baked into the recording.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kinefly2.analysis import calibrate

LOG = 'logs/260819/260819_XHCS_017_YAMV/260819_XHCS_017_YAMV_flystate.h5'   # <- your recording

log = calibrate.load_log(LOG)
print(f"{len(log['merged/seq'])} rows")
print('columns:', sorted(k for k in log if k.startswith('front/') or k.startswith('bottom/')))

## 1. Pick the reference yaw

**Option A — the bottom camera** (needs both cameras running). Its head angle
is a true in-plane rotation, already in radians.

In [ ]:
shift = log['front/head_dx_corr']      # yaw channel, px
yaw   = log['bottom/head_angle']       # reference yaw, rad

**Option B — a rotation stage.** Load your own (time, angle) ground truth and
interpolate it onto the log's clock. This is the only reference that also
checks the bottom camera's own yaw angle, which nothing else does.

```python
ref_t, ref_deg = np.loadtxt('stage_angles.csv', delimiter=',', unpack=True)
yaw = np.interp(log['merged/t_wall'], ref_t, np.radians(ref_deg))
```

**Option C — restrict to stimulus epochs** (once the arena is running):

```python
mask = calibrate.epochs_from_events(log, 'stim_on')
```

## 2. Fit, and check whether the fit may be trusted

`check_fit` refuses recordings that cannot support a constant. A lever arm
fitted from a fly that barely moved its head is not a measurement — it is noise
with units, and it would silently rescale every yaw and pitch number derived
from it afterwards.

In [ ]:
fit = calibrate.fit_lever(shift, yaw)          # add mask=mask to restrict to epochs
ok, why = calibrate.check_fit(fit)
print(calibrate.report(fit, ok, why))

In [ ]:
m = np.isfinite(shift) & np.isfinite(yaw)
yc = yaw[m] - np.median(yaw[m])
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(np.degrees(yc), shift[m], '.', ms=1, alpha=.25)
if np.isfinite(fit['lever_px']):
    g = np.linspace(np.degrees(yc).min(), np.degrees(yc).max(), 100)
    plt.plot(g, fit['lever_px'] * np.sin(np.radians(g)) + fit['offset_px'], 'r-', lw=2)
plt.xlabel('reference yaw (deg)'); plt.ylabel('front head_dx_corr (px)')
plt.title(f"lever = {fit['lever_px']:.1f} px/rad, r2 = {fit['r2']:.3f}")

plt.subplot(1, 2, 2)
t = log['merged/t_host'][m]; t = t - t[0]
plt.plot(t, np.degrees(yc), lw=.7, label='reference yaw (deg)')
plt.plot(t, shift[m], lw=.7, label='front shift (px)')
plt.xlabel('time (s)'); plt.legend(); plt.title('do they move together?')
plt.tight_layout()

### If it was rejected

The pixel channels are still perfectly valid — only the *conversion to angles*
is withheld. The usual reason is too little head yaw in the recording. For
reference, the 2026-08-19 cam1/cam2 pair has a bottom-camera yaw std of 0.35°
against 0.22° of frame-to-frame noise; that is nowhere near enough. Record
again with an optomotor stimulus, or use a rotation stage.

## 3. Apply it — the same constant for both channels

In [ ]:
if ok:
    lever = fit['lever_px']
    yaw_deg   = np.degrees(calibrate.apply_lever(log['front/head_dx_corr'], lever))
    pitch_deg = np.degrees(calibrate.apply_lever(log['front/head_dy_corr'], lever))
    roll_deg  = np.degrees(log['front/head_angle'])

    for name, v in (('roll', roll_deg), ('yaw', yaw_deg), ('pitch', pitch_deg)):
        print(f'{name:6s} std {np.nanstd(v):5.2f} deg   '
              f'range {np.nanpercentile(v,1):+6.1f} .. {np.nanpercentile(v,99):+6.1f}')

    t = log['merged/t_host'] - log['merged/t_host'][0]
    fig, ax = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
    for a, (name, v) in zip(ax, (('roll', roll_deg), ('yaw', yaw_deg), ('pitch', pitch_deg))):
        a.plot(t, v, lw=.6); a.set_ylabel(name + ' (deg)'); a.axhline(0, color='k', lw=.4)
    ax[-1].set_xlabel('time (s)'); plt.tight_layout()
else:
    print('Not calibrated - keep using the pixel channels.')

## 4. Make it permanent

Put the constant in the **head:** section of `config/camera_front.yaml`:

```yaml
tracker:
  head:
    tracker: landmarks
    lever_px: 92.4        # <- from this notebook
```

Kinefly2 then logs `front/head_yaw` and `front/head_pitch` in radians and shows
degrees in the GUI. Without it those columns stay NaN rather than a guess.

**The constant is specific to this fly, this magnification and this landmark
set.** Re-fit it after moving the camera, changing the lens or refocusing. If
you want it to travel across flies, divide by a size proxy measured in the same
image (eye-to-eye distance in px) and rescale per fly.

Sanity check: `lever_px` should equal the head's anterior–posterior depth in
pixels. Divide by your px/mm to confirm it lands near a few tenths of a
millimetre. If it comes out at several millimetres, the fit found something
other than head yaw.